# Approach D: CH4Net Retraining — Focal + Dice Loss

**What changed vs. Approach C (baseline F1 = 0.2733):**
1. **Loss**: `BCEWithLogitsLoss` → `FocalLoss + DiceLoss` (0.5 each)
2. **Augmentation**: center-crop only → random crop + random H/V flip during training

Everything else (architecture, div_factor=8, optimizer, scheduler) is identical for a clean A/B comparison.

**Runtime**: ~2h on Colab T4 for 100 epochs. Set `EPOCHS = 50` for a quick check first.

---
**Before running**: Make sure GPU is enabled → Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Copy dataset from Drive to local SSD (fast I/O during training) ─────────
# Only needs to run once per Colab session (~16 min for 9GB)
import os, shutil

DRIVE_DATA = '/content/drive/MyDrive/ch4net_official'   # adjust if your path differs
LOCAL_DATA = '/content/ch4net_official'

if not os.path.exists(LOCAL_DATA):
    print('Copying dataset to local SSD...')
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print('Done.')
else:
    print('Dataset already on local SSD, skipping copy.')

# Verify
for split in ['train', 'val']:
    n = len(os.listdir(os.path.join(LOCAL_DATA, split, 's2')))
    print(f'  {split}: {n} files')

In [ ]:
# ── 3. Config — edit these before running ──────────────────────────────────────
DATA_DIR    = '/content/ch4net_official'
OUT_WEIGHTS = '/content/drive/MyDrive/weights/ch4net_div8_focal_dice.pth'  # saves direct to Drive
EPOCHS      = 100     # 50 for a quick check, 100 for full run
BATCH_SIZE  = 16
LR          = 1e-3
DIV_FACTOR  = 8       # paper spec — do not change
THRESHOLD   = 0.5     # val F1 threshold

# Loss weights
DICE_W      = 0.5
FOCAL_W     = 0.5
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

CROP_H, CROP_W = 160, 160

In [ ]:
# ── 4. Imports ─────────────────────────────────────────────────────────────────
import glob, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 5. Augmentation helpers ────────────────────────────────────────────────────
def random_crop(img, lbl, h, w):
    """Random crop from (C,H,W) arrays. Pads if smaller than target."""
    _, ih, iw = img.shape
    if ih < h or iw < w:
        ph = max(0, h - ih); pw = max(0, w - iw)
        img = np.pad(img, [(0,0),(ph//2, ph-ph//2),(pw//2, pw-pw//2)])
        lbl = np.pad(lbl, [(0,0),(ph//2, ph-ph//2),(pw//2, pw-pw//2)])
        return img, lbl
    top  = random.randint(0, ih - h)
    left = random.randint(0, iw - w)
    return img[:, top:top+h, left:left+w], lbl[:, top:top+h, left:left+w]

def center_crop_pad(arr, th, tw):
    """Deterministic center-crop or zero-pad (C,H,W) to (C,th,tw)."""
    _, h, w = arr.shape
    if h >= th:
        s = (h-th)//2; arr = arr[:, s:s+th, :]
    else:
        p = th-h; arr = np.pad(arr, [(0,0),(p//2,p-p//2),(0,0)])
    _, h2, w2 = arr.shape
    if w2 >= tw:
        s = (w2-tw)//2; arr = arr[:, :, s:s+tw]
    else:
        p = tw-w2; arr = np.pad(arr, [(0,0),(0,0),(p//2,p-p//2)])
    return arr

In [ ]:
# ── 6. Dataset ─────────────────────────────────────────────────────────────────
class CH4NetDataset(Dataset):
    def __init__(self, split_dir, augment=False):
        self.s2    = sorted(glob.glob(os.path.join(split_dir, 's2',    '*.npy')))
        self.lbls  = sorted(glob.glob(os.path.join(split_dir, 'label', '*.npy')))
        self.aug   = augment
        assert len(self.s2) == len(self.lbls) and len(self.s2) > 0
        n_pos = sum(1 for p in self.lbls if np.load(p).max() > 0) if len(self.lbls) < 500 else '?'
        print(f'  {os.path.basename(split_dir):5s}: {len(self.s2):5d} samples'
              + (f'  ({n_pos} positive)' if n_pos != '?' else '')
              + (' [aug]' if augment else ''))

    def __len__(self): return len(self.s2)

    def __getitem__(self, idx):
        img = np.load(self.s2[idx]).copy().astype(np.float32) / 255.0
        img = np.clip(img.transpose(2,0,1), 0.0, 1.0)
        lbl = np.load(self.lbls[idx]).copy().astype(np.float32)
        lbl = (lbl > 0).astype(np.float32)[np.newaxis]

        if self.aug:
            img, lbl = random_crop(img, lbl, CROP_H, CROP_W)
            if random.random() > 0.5:
                img = img[:, :, ::-1].copy(); lbl = lbl[:, :, ::-1].copy()
            if random.random() > 0.5:
                img = img[:, ::-1, :].copy(); lbl = lbl[:, ::-1, :].copy()
        else:
            img = center_crop_pad(img, CROP_H, CROP_W)
            lbl = center_crop_pad(lbl, CROP_H, CROP_W)

        return torch.from_numpy(img.copy()), torch.from_numpy(lbl.copy())

print('Loading datasets...')
train_ds = CH4NetDataset(os.path.join(DATA_DIR, 'train'), augment=True)
val_ds   = CH4NetDataset(os.path.join(DATA_DIR, 'val'),   augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# ── 7. Loss functions ──────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs   = torch.sigmoid(logits)
        targets = (targets > 0.5).float()
        probs   = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        intersection = (probs * targets).sum(dim=1)
        dice = (2.*intersection + self.smooth) / (probs.sum(dim=1) + targets.sum(dim=1) + self.smooth)
        return 1. - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, targets):
        targets = (targets > 0.5).float()
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs   = torch.sigmoid(logits)
        pt      = targets * probs + (1.-targets) * (1.-probs)
        alpha_t = targets * self.alpha + (1.-targets) * (1.-self.alpha)
        return (alpha_t * (1.-pt)**self.gamma * bce).mean()

class FocalDiceLoss(nn.Module):
    def __init__(self, dice_w=0.5, focal_w=0.5, gamma=2.0, alpha=0.25):
        super().__init__()
        self.dice_w = dice_w; self.focal_w = focal_w
        self.dice   = DiceLoss()
        self.focal  = FocalLoss(gamma=gamma, alpha=alpha)
    def forward(self, logits, targets):
        return self.dice_w*self.dice(logits,targets) + self.focal_w*self.focal(logits,targets)

print(f'Loss: Focal(γ={FOCAL_GAMMA}, α={FOCAL_ALPHA})×{FOCAL_W} + Dice×{DICE_W}')

In [ ]:
# ── 8. Model ───────────────────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        dy, dx = x2.size(2)-x1.size(2), x2.size(3)-x1.size(3)
        x1 = F.pad(x1, [dx//2, dx-dx//2, dy//2, dy-dy//2])
        return self.conv(torch.cat([x2, x1], dim=1))

class Unet(nn.Module):
    def __init__(self, in_channels=12, div_factor=8):
        super().__init__()
        d = div_factor
        self.inc   = DoubleConv(in_channels, 64//d)
        self.down1 = Down(64//d,  128//d)
        self.down2 = Down(128//d, 256//d)
        self.down3 = Down(256//d, 512//d)
        self.down4 = Down(512//d, 512//d)
        self.up1   = Up(1024//d,  256//d)
        self.up2   = Up(512//d,   128//d)
        self.up3   = Up(256//d,    64//d)
        self.up4   = Up(128//d,   128//d)
        self.out   = nn.Conv2d(128//d, 1, kernel_size=1)
    def forward(self, x):
        x1=self.inc(x); x2=self.down1(x1); x3=self.down2(x2)
        x4=self.down3(x3); x5=self.down4(x4)
        x=self.up1(x5,x4); x=self.up2(x,x3); x=self.up3(x,x2); x=self.up4(x,x1)
        return self.out(x)

model = Unet(in_channels=12, div_factor=DIV_FACTOR).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: div_factor={DIV_FACTOR}, params={n_params:,}')

In [ ]:
# ── 9. Training loop ───────────────────────────────────────────────────────────
def pixel_f1(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    tgts  = (targets > 0.5).float()
    tp = (preds * tgts).sum()
    fp = (preds * (1-tgts)).sum()
    fn = ((1-preds) * tgts).sum()
    return (2*tp / (2*tp + fp + fn + 1e-8)).item()

criterion = FocalDiceLoss(dice_w=DICE_W, focal_w=FOCAL_W, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)
optimiser = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimiser, mode='max', factor=0.5, patience=5)

os.makedirs(os.path.dirname(OUT_WEIGHTS), exist_ok=True)
best_val_f1 = 0.0

print(f"{'Epoch':>6}  {'Train Loss':>12}  {'Val Loss':>10}  {'Val F1':>10}  {'Saved?':>7}")
print('─' * 55)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimiser.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimiser.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    # Validate
    model.eval()
    val_loss = val_f1 = 0.0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits   = model(imgs)
            val_loss += criterion(logits, lbls).item() * imgs.size(0)
            val_f1   += pixel_f1(logits, lbls, THRESHOLD) * imgs.size(0)
    val_loss /= len(val_ds)
    val_f1   /= len(val_ds)

    scheduler.step(val_f1)
    is_best = val_f1 > best_val_f1
    if is_best:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), OUT_WEIGHTS)

    print(f"{epoch:>6}  {train_loss:>12.6f}  {val_loss:>10.6f}  "
          f"{val_f1:>10.4f}  {'  ✓' if is_best else ''}")

print(f'\nDone. Best val F1: {best_val_f1:.4f}  (baseline approach_c: 0.2733)')
print(f'Weights saved to: {OUT_WEIGHTS}')